# AIFS ENS v2 heat-hazard forecast pipeline

This notebook converts six-hourly AIFS ENS v2 forecasts into the same local-solar daily mean/min/max statistics used for ERA5, applies the fixed ERA5 1991–2020 q95 thresholds, creates ensemble probabilities for hot days and 2-/3-day event onsets, matches ERA5 observations by valid date, and computes first-pass lead-dependent RMSE and Brier scores.

Outputs are organized under:

```text
/net/monsoon/kylehall/ERA5/heat_extremes_aifs_ens_v2/
├── daily/
├── probabilities/
├── matched_observations/
├── scores/
├── diagnostics/
└── figures/
```

Start with one year as a smoke test, then expand `YEARS`.

In [1]:
from __future__ import annotations

import json
import re
from collections.abc import Iterable
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dask.diagnostics import ProgressBar
from xclim.core.calendar import resample_doy

xr.set_options(keep_attrs=True)


In [2]:
AIFS_ROOT = Path("/net/monsoon/marchakitus/AIFS/v2p0/combined/forecasts_AIFS_ENS_v2")
ERA5_ROOT = Path("/net/monsoon/kylehall/ERA5/heat_extremes_climatology")
OUTPUT_ROOT = Path("/net/monsoon/kylehall/ERA5/heat_extremes_aifs_ens_v2")

DAILY_DIR = OUTPUT_ROOT / "daily"
PROBABILITY_DIR = OUTPUT_ROOT / "probabilities"
OBSERVATION_DIR = OUTPUT_ROOT / "matched_observations"
SCORE_DIR = OUTPUT_ROOT / "scores"
DIAGNOSTIC_DIR = OUTPUT_ROOT / "diagnostics"
FIGURE_DIR = OUTPUT_ROOT / "figures"

for d in (DAILY_DIR, PROBABILITY_DIR, OBSERVATION_DIR, SCORE_DIR, DIAGNOSTIC_DIR, FIGURE_DIR):
    d.mkdir(parents=True, exist_ok=True)

YEARS = (2022, 2023, 2024, 2025)       # smoke test first
MAX_DAYS = 15
STATISTICS = ("mean", "min", "max")
PRIMARY_PERCENTILE = 95.0
EVENT_DURATIONS = (2, 3)
SAVE_MEMBER_HAZARDS = False

OPEN_CHUNKS = {
    "time": 1,
    "number": 26,
    "prediction_timedelta": 24,
    "latitude": 180,
    "longitude": 180,
}
DAILY_CHUNKS = {
    "time": 1,
    "number": 26,
    "forecast_day": 8,
    "latitude": 180,
    "longitude": 180,
}
PROBABILITY_CHUNKS = {
    "time": 1,
    "forecast_day": 8,
    "latitude": 180,
    "longitude": 180,
}

print(OUTPUT_ROOT)


/net/monsoon/kylehall/ERA5/heat_extremes_aifs_ens_v2


## Open and normalize AIFS ENS v2

In [3]:
DEFAULT_VARIABLES = ("2d", "2t", "tp")


def forecast_store_year(path: Path) -> int | None:
    match = re.search(r"(?:19|20)\d{2}", path.name)
    return None if match is None else int(match.group())


def open_aifs_ens_v2(
    root: str | Path = AIFS_ROOT,
    *,
    years: Iterable[int] | None = None,
    variables: Iterable[str] = DEFAULT_VARIABLES,
    chunks: dict[str, int] | None = None,
) -> xr.Dataset:
    """Open one-store-per-initialization AIFS ENS v2 forecasts."""
    root = Path(root)
    paths = sorted(root.glob("*.zarr"))

    if years is not None:
        wanted_years = {int(y) for y in years}
        paths = [p for p in paths if forecast_store_year(p) in wanted_years]
        selection = f"years {sorted(wanted_years)}"
    else:
        selection = "all years"

    if not paths:
        raise FileNotFoundError(f"No AIFS stores under {root} for {selection}")

    requested = tuple(variables)

    def preprocess(ds: xr.Dataset) -> xr.Dataset:
        missing = [name for name in requested if name not in ds]
        if missing:
            raise KeyError(f"AIFS store is missing variables: {missing}")
        return ds[list(requested)]

    ds = xr.open_mfdataset(
        [str(p) for p in paths],
        engine="zarr",
        combine="nested",
        concat_dim="time",
        preprocess=preprocess,
        chunks=OPEN_CHUNKS if chunks is None else chunks,
        parallel=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        join="override",
        combine_attrs="override",
        consolidated=None,
    )

    rename = {
        "2d": "2m_dewpoint_temperature",
        "2t": "2m_temperature",
        "tp": "total_precipitation",
        "lat": "latitude",
        "lon": "longitude",
    }
    ds = ds.rename({
        old: new for old, new in rename.items()
        if old in ds.variables or old in ds.dims
    })

    required = {"time", "number", "prediction_timedelta", "latitude", "longitude"}
    missing = required.difference(ds.dims)
    if missing:
        raise ValueError(f"Missing AIFS dimensions: {sorted(missing)}")

    ds = ds.sortby("time").sortby("prediction_timedelta")
    if np.any(np.diff(ds.longitude.values) <= 0):
        ds = ds.sortby("longitude")

    return ds


In [4]:
aifs = open_aifs_ens_v2(
    years=YEARS,
    variables=("2t",),
    chunks=OPEN_CHUNKS,
)

print(aifs)
print("Initialization hours:", np.unique(aifs.time.dt.hour.values))
print("Lead range:", aifs.prediction_timedelta.values[[0, -1]])


/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/zarr/codecs/numcodecs/_codecs.py:163: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/zarr/codecs/numcodecs/_codecs.py:163: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/zarr/codecs/numcodecs/_codecs.py:163: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/zarr/codecs/numcodecs/_codecs.py:163: ZarrUserWarning: Numcodecs codecs are not in 

<xarray.Dataset> Size: 8TB
Dimensions:               (time: 364, number: 26, prediction_timedelta: 200,
                           latitude: 721, longitude: 1440)
Coordinates:
  * time                  (time) datetime64[ns] 3kB 2022-01-01 ... 2025-12-31
  * number                (number) int64 208B 0 1 2 3 4 5 ... 20 21 22 23 24 25
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 2kB 06:00:00...
  * latitude              (latitude) float64 6kB 90.0 89.75 ... -89.75 -90.0
  * longitude             (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
Data variables:
    2m_temperature        (time, number, prediction_timedelta, latitude, longitude) float32 8TB dask.array<chunksize=(1, 26, 24, 90, 180), meta=np.ndarray>
Initialization hours: [0]
Lead range: [  21600000000000 4320000000000000]


## Local-solar daily aggregation

In [5]:
def longitude_band_specs(longitudes: xr.DataArray):
    values = np.asarray(longitudes.values)
    if np.any(np.diff(values) <= 0):
        raise ValueError("Longitude must be strictly increasing.")

    if values.min() >= 0 and values.max() > 180:
        return (
            (0.0, 45.0, 0),
            (45.0, 135.0, 6),
            (135.0, 180.0, 12),
            (180.0, 225.0, -12),
            (225.0, 315.0, -6),
            (315.0, 360.0, 0),
        )
    if values.min() < 0 and values.max() <= 180:
        return (
            (-180.0, -135.0, -12),
            (-135.0, -45.0, -6),
            (-45.0, 45.0, 0),
            (45.0, 135.0, 6),
            (135.0, 180.0, 12),
        )
    raise ValueError("Unsupported longitude convention.")


def select_half_open(da, start, stop, lon_dim="longitude"):
    return da.sel({lon_dim: slice(start, np.nextafter(stop, -np.inf))})


def aggregate_init_hour_group(
    temperature: xr.DataArray,
    *,
    init_hour: int,
    offset_hours: int,
    statistic: str,
    init_dim: str = "time",
    step_dim: str = "prediction_timedelta",
) -> xr.DataArray:
    if statistic not in {"mean", "min", "max"}:
        raise ValueError("statistic must be mean, min, or max")

    steps = temperature[step_dim]
    step_hours = (steps / np.timedelta64(1, "h")).astype(int)
    local_hour = (init_hour + step_hours + offset_hours) % 24
    midnight = np.flatnonzero(local_hour.values == 0)

    if midnight.size == 0:
        raise ValueError("No local-midnight step found.")

    aligned = temperature.isel({step_dim: slice(int(midnight[0]), None)})
    n_samples = (aligned.sizes[step_dim] // 4) * 4
    if n_samples == 0:
        raise ValueError("No complete local day available.")

    aligned = aligned.isel({step_dim: slice(0, n_samples)})
    grouped = aligned.coarsen(
        {step_dim: 4},
        boundary="exact",
        coord_func={step_dim: "min"},
    )
    daily = getattr(grouped, statistic)()

    first_steps = aligned[step_dim].isel({step_dim: slice(0, None, 4)})
    n_days = first_steps.size

    daily = daily.rename({step_dim: "forecast_day"})
    daily = daily.assign_coords(forecast_day=np.arange(n_days, dtype=np.int16))

    valid_date = (
        temperature[init_dim] + first_steps + np.timedelta64(offset_hours, "h")
    ).dt.floor("D")
    valid_date = valid_date.rename({step_dim: "forecast_day"})
    valid_date = valid_date.assign_coords(forecast_day=daily.forecast_day)

    return daily.assign_coords(valid_date=valid_date)


def local_solar_daily_forecast_stat(
    temperature: xr.DataArray,
    *,
    statistic: str,
    max_days: int | None = None,
) -> xr.DataArray:
    """Aggregate AIFS members into approximate local-solar daily statistics."""
    if max_days is not None:
        temperature = temperature.where(
            temperature.prediction_timedelta < np.timedelta64(max_days, "D"),
            drop=True,
        )

    init_hours = np.unique(temperature.time.dt.hour.values)
    band_results = []

    for start, stop, offset in longitude_band_specs(temperature.longitude):
        band = select_half_open(temperature, start, stop)
        if band.sizes.get("longitude", 0) == 0:
            continue

        hour_results = []
        for hour in init_hours:
            hour_band = band.sel(time=band.time.dt.hour == int(hour))
            if hour_band.sizes["time"] == 0:
                continue
            hour_results.append(
                aggregate_init_hour_group(
                    hour_band,
                    init_hour=int(hour),
                    offset_hours=offset,
                    statistic=statistic,
                )
            )

        band_daily = xr.concat(
            hour_results,
            dim="time",
            join="outer",
            coords="minimal",
            compat="override",
        ).sortby("time")

        band_daily = band_daily.assign_coords(
            local_solar_offset_hours=xr.full_like(
                band_daily.longitude, offset, dtype=np.int8
            )
        )
        band_results.append(band_daily)

    daily = xr.concat(
        band_results,
        dim="longitude",
        join="inner",
        coords="minimal",
        compat="override",
    ).sortby("longitude")

    daily.name = f"t2m_daily_{statistic}"
    daily.attrs.update(
        source="AIFS ENS v2 six-hourly forecasts",
        daily_statistic=statistic,
        daily_time_basis="Approximate local-solar day using six-hour UTC-offset bands",
    )
    return daily


## Utilities

In [8]:
def clear_chunk_encoding(obj):
    obj = obj.copy()
    if isinstance(obj, xr.Dataset):
        for name in obj.variables:
            obj[name].encoding.pop("chunks", None)
            obj[name].encoding.pop("preferred_chunks", None)
    else:
        obj.encoding.pop("chunks", None)
        obj.encoding.pop("preferred_chunks", None)
    return obj


def write_zarr(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(obj, xr.DataArray):
        if obj.name is None:
            raise ValueError("DataArray must be named.")
        obj = obj.to_dataset()
    obj = clear_chunk_encoding(obj)
    with ProgressBar():
        obj.to_zarr(
            path,
            mode="w",
            consolidated=True,
           # zarr_format=2,
        )


def heatwave_start_mask(hot, min_duration, day_dim="forecast_day"):
    if min_duration < 1:
        raise ValueError("min_duration must be at least one")
    hot = hot.fillna(False).astype(bool)
    qualifies = xr.concat(
        [hot.shift({day_dim: -lag}, fill_value=False)
         for lag in range(min_duration)],
        dim="_duration_check",
    ).all("_duration_check")
    previous = hot.shift({day_dim: 1}, fill_value=False)
    return qualifies & ~previous


def area_weighted_mean(da):
    weights = np.cos(np.deg2rad(da.latitude))
    return da.weighted(weights).mean(("latitude", "longitude"))


def normalize_longitude_like(source, target_longitude):
    target_uses_360 = (
        float(target_longitude.min()) >= 0
        and float(target_longitude.max()) > 180
    )
    lon = source.longitude % 360 if target_uses_360 else (
        (source.longitude + 180) % 360
    ) - 180
    return source.assign_coords(longitude=lon).sortby("longitude")


def map_to_forecast_grid(source, forecast, method):
    source = normalize_longitude_like(source, forecast.longitude)
    if (
        np.array_equal(source.latitude.values, forecast.latitude.values)
        and np.array_equal(source.longitude.values, forecast.longitude.values)
    ):
        return source
    return source.interp(
        latitude=forecast.latitude,
        longitude=forecast.longitude,
        method=method,
    )


def expand_dayofyear_threshold(threshold, valid_date):
    """Use xclim calendar handling on stacked forecast valid dates."""
    stacked = valid_date.stack(forecast_case=("time", "forecast_day"))
    template = xr.DataArray(
        np.zeros(stacked.size, dtype=np.float32),
        dims=("forecast_case",),
        coords={
            "forecast_case": stacked.forecast_case,
            "time": ("forecast_case", stacked.values),
        },
    )
    expanded = resample_doy(threshold, template)
    return expanded.unstack("forecast_case").transpose(
        "time", "forecast_day", ...
    )


def match_observation_by_valid_date(observation, valid_date):
    stacked = valid_date.stack(forecast_case=("time", "forecast_day"))
    indexer = xr.DataArray(
        stacked.values,
        dims=("forecast_case",),
        coords={"forecast_case": stacked.forecast_case},
    )
    matched = observation.sel(time=indexer)
    return matched.unstack("forecast_case").transpose(
        "time", "forecast_day", ...
    )


## 1. Build and checkpoint local-solar daily forecasts

In [9]:
daily_paths = {}

for statistic in STATISTICS:
    path = DAILY_DIR / (
        f"aifs_ens_v2_t2m_daily_{statistic}_{min(YEARS)}_{max(YEARS)}.zarr"
    )
    print(f"\nBuilding {statistic}: {path}")

    daily = local_solar_daily_forecast_stat(
        aifs["2m_temperature"],
        statistic=statistic,
        max_days=MAX_DAYS,
    ).chunk({
        dim: size for dim, size in DAILY_CHUNKS.items()
        if dim in aifs["2m_temperature"].dims or dim == "forecast_day"
    })

    if "local_solar_offset_hours" in daily.coords:
        daily = daily.assign_coords(
            local_solar_offset_hours=(
                "longitude",
                daily.local_solar_offset_hours.values,
            )
        )

    print(daily)
    print("Chunks:", daily.chunks)
    write_zarr(daily, path)
    daily_paths[statistic] = path
    



Building mean: /net/monsoon/kylehall/ERA5/heat_extremes_aifs_ens_v2/daily/aifs_ens_v2_t2m_daily_mean_2022_2025.zarr
<xarray.DataArray 't2m_daily_mean' (time: 364, number: 26, forecast_day: 14,
                                    latitude: 721, longitude: 1440)> Size: 550GB
dask.array<rechunk-merge, shape=(364, 26, 14, 721, 1440), dtype=float32, chunksize=(1, 26, 8, 180, 180), chunktype=numpy.ndarray>
Coordinates:
  * time                      (time) datetime64[ns] 3kB 2022-01-01 ... 2025-1...
  * number                    (number) int64 208B 0 1 2 3 4 5 ... 21 22 23 24 25
  * forecast_day              (forecast_day) int16 28B 0 1 2 3 4 ... 10 11 12 13
    valid_date                (time, forecast_day) datetime64[ns] 41kB dask.array<chunksize=(1, 8), meta=np.ndarray>
  * latitude                  (latitude) float64 6kB 90.0 89.75 ... -89.75 -90.0
  * longitude                 (longitude) float64 12kB 0.0 0.25 ... 359.5 359.8
    local_solar_offset_hours  (longitude) int8 1kB 0 0 0 0 0 

/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


[                                        ] | 2% Completed | 317.76 ss

IOStream.flush timed out


[##                                      ] | 6% Completed | 11m 21ss


KeyboardInterrupt: 

## 2. Validate dates

In [ ]:
validation_rows = []

for statistic, path in daily_paths.items():
    daily = xr.open_zarr(path, consolidated=True, chunks={})[
        f"t2m_daily_{statistic}"
    ]
    increments = np.unique(
        daily.valid_date.diff("forecast_day").values.astype("timedelta64[D]")
    )
    assert np.all(increments == np.timedelta64(1, "D")), increments

    validation_rows.append({
        "statistic": statistic,
        "initializations": daily.sizes["time"],
        "members": daily.sizes["number"],
        "complete_local_days": daily.sizes["forecast_day"],
        "first_valid_date": str(daily.valid_date.min().compute().item()),
        "last_valid_date": str(daily.valid_date.max().compute().item()),
    })

validation = pd.DataFrame(validation_rows)
validation.to_csv(DIAGNOSTIC_DIR / "daily_forecast_validation.csv", index=False)
display(validation)


## 3. Apply ERA5 q95 thresholds and create ensemble probabilities

In [ ]:
probability_paths = {}

for statistic, daily_path in daily_paths.items():
    daily = xr.open_zarr(daily_path, consolidated=True, chunks={})[
        f"t2m_daily_{statistic}"
    ]

    threshold_path = ERA5_ROOT / "thresholds" / (
        f"t2m_daily_{statistic}_percentiles_1991_2020.zarr"
    )
    threshold = xr.open_zarr(
        threshold_path, consolidated=True, chunks={}
    )[f"t2m_daily_{statistic}_calendar_day_percentile"].sel(
        percentiles=PRIMARY_PERCENTILE, drop=True
    )

    threshold = map_to_forecast_grid(threshold, daily, method="linear")
    q95 = expand_dayofyear_threshold(threshold, daily.valid_date)

    member_hot = (daily > q95).rename("member_hot_day_q95")
    variables = {
        "hot_day_q95_probability": member_hot.mean("number"),
    }

    member_variables = {"member_hot_day_q95": member_hot}

    for duration in EVENT_DURATIONS:
        member_start = heatwave_start_mask(member_hot, duration)
        variables[f"heatwave_start_q95_{duration}d_probability"] = (
            member_start.mean("number")
        )
        member_variables[f"member_heatwave_start_q95_{duration}d"] = member_start

    probabilities = xr.Dataset(variables).chunk(PROBABILITY_CHUNKS)
    probabilities.attrs.update(
        source="AIFS ENS v2",
        threshold="ERA5 1991-2020 calendar-day q95",
        statistic=statistic,
        ensemble_size=daily.sizes["number"],
    )

    path = PROBABILITY_DIR / (
        f"aifs_ens_v2_t2m_daily_{statistic}_q95_probabilities_"
        f"{min(YEARS)}_{max(YEARS)}.zarr"
    )
    write_zarr(probabilities, path)
    probability_paths[statistic] = path

    if SAVE_MEMBER_HAZARDS:
        member_path = PROBABILITY_DIR / (
            f"aifs_ens_v2_t2m_daily_{statistic}_q95_member_hazards_"
            f"{min(YEARS)}_{max(YEARS)}.zarr"
        )
        write_zarr(xr.Dataset(member_variables).chunk(DAILY_CHUNKS), member_path)


## 4. Match ERA5 observations to forecast valid dates

In [ ]:
observation_paths = {}

for statistic, daily_path in daily_paths.items():
    forecast = xr.open_zarr(daily_path, consolidated=True, chunks={})[
        f"t2m_daily_{statistic}"
    ]

    era5_temperature = xr.open_zarr(
        ERA5_ROOT / "daily" / f"t2m_daily_{statistic}.zarr",
        consolidated=True,
        chunks={},
    )[f"t2m_daily_{statistic}"]

    era5_hazards = xr.open_zarr(
        ERA5_ROOT / "hazards" / f"t2m_daily_{statistic}_q95_hazards.zarr",
        consolidated=True,
        chunks={},
    )

    era5_temperature = map_to_forecast_grid(
        era5_temperature, forecast, method="linear"
    )

    matched = {
        "observed_temperature": match_observation_by_valid_date(
            era5_temperature, forecast.valid_date
        )
    }

    for name in (
        "hot_day_q95",
        "heatwave_start_q95_2d",
        "heatwave_start_q95_3d",
    ):
        field = map_to_forecast_grid(
            era5_hazards[name].astype(np.float32),
            forecast,
            method="nearest",
        )
        matched[f"observed_{name}"] = (
            match_observation_by_valid_date(field, forecast.valid_date) > 0.5
        )

    matched = xr.Dataset(matched).assign_coords(valid_date=forecast.valid_date)
    matched = matched.chunk(PROBABILITY_CHUNKS)

    path = OBSERVATION_DIR / (
        f"era5_matched_to_aifs_daily_{statistic}_"
        f"{min(YEARS)}_{max(YEARS)}.zarr"
    )
    write_zarr(matched, path)
    observation_paths[statistic] = path


## 5. First-pass global lead-dependent scores

In [ ]:
rows = []

for statistic in STATISTICS:
    forecast = xr.open_zarr(
        daily_paths[statistic], consolidated=True, chunks={}
    )[f"t2m_daily_{statistic}"]
    probabilities = xr.open_zarr(
        probability_paths[statistic], consolidated=True, chunks={}
    )
    observations = xr.open_zarr(
        observation_paths[statistic], consolidated=True, chunks={}
    )

    ensemble_mean = forecast.mean("number")
    rmse = np.sqrt(
        area_weighted_mean(
            (ensemble_mean - observations.observed_temperature) ** 2
        ).mean("time")
    ).compute()

    event_pairs = {
        "hot_day_q95": (
            probabilities.hot_day_q95_probability,
            observations.observed_hot_day_q95,
        ),
        "heatwave_start_q95_2d": (
            probabilities.heatwave_start_q95_2d_probability,
            observations.observed_heatwave_start_q95_2d,
        ),
        "heatwave_start_q95_3d": (
            probabilities.heatwave_start_q95_3d_probability,
            observations.observed_heatwave_start_q95_3d,
        ),
    }

    summaries = {}
    for event, (p, y) in event_pairs.items():
        y = y.astype(np.float32)
        summaries[event] = {
            "brier": area_weighted_mean((p - y) ** 2).mean("time").compute(),
            "mean_probability": area_weighted_mean(p).mean("time").compute(),
            "observed_frequency": area_weighted_mean(y).mean("time").compute(),
        }

    for day in forecast.forecast_day.values:
        row = {
            "statistic": statistic,
            "forecast_day": int(day),
            "rmse": float(rmse.sel(forecast_day=day)),
        }
        for event, summary in summaries.items():
            pbar = float(summary["mean_probability"].sel(forecast_day=day))
            ybar = float(summary["observed_frequency"].sel(forecast_day=day))
            row[f"{event}_brier_score"] = float(
                summary["brier"].sel(forecast_day=day)
            )
            row[f"{event}_mean_probability"] = pbar
            row[f"{event}_observed_frequency"] = ybar
            row[f"{event}_frequency_bias"] = pbar - ybar
        rows.append(row)

scores = pd.DataFrame(rows)
scores.to_csv(
    SCORE_DIR / f"global_lead_scores_{min(YEARS)}_{max(YEARS)}.csv",
    index=False,
)
display(scores.head(12))


## 6. Diagnostic figures

In [ ]:
for statistic in STATISTICS:
    subset = scores[scores.statistic == statistic]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(subset.forecast_day, subset.rmse, marker="o")
    ax.set(
        xlabel="Forecast day index",
        ylabel="Area-weighted RMSE",
        title=f"AIFS ENS v2 daily-{statistic} temperature RMSE",
    )
    ax.grid(alpha=0.3)
    fig.savefig(
        FIGURE_DIR / f"rmse_daily_{statistic}.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    for event, label in (
        ("hot_day_q95", "Hot day"),
        ("heatwave_start_q95_2d", "2+ day onset"),
        ("heatwave_start_q95_3d", "3+ day onset"),
    ):
        ax.plot(
            subset.forecast_day,
            subset[f"{event}_brier_score"],
            marker="o",
            label=label,
        )
    ax.set(
        xlabel="Forecast day index",
        ylabel="Area-weighted Brier score",
        title=f"AIFS ENS v2 daily-{statistic} heat-hazard score",
    )
    ax.grid(alpha=0.3)
    ax.legend()
    fig.savefig(
        FIGURE_DIR / f"brier_daily_{statistic}.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


## 7. Save metadata

In [ ]:
metadata = {
    "aifs_root": str(AIFS_ROOT),
    "era5_root": str(ERA5_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "years": list(YEARS),
    "max_days": MAX_DAYS,
    "statistics": list(STATISTICS),
    "primary_percentile": PRIMARY_PERCENTILE,
    "event_durations": list(EVENT_DURATIONS),
    "ensemble_size": int(aifs.sizes["number"]),
    "initializations": int(aifs.sizes["time"]),
    "initialization_hours": [
        int(x) for x in np.unique(aifs.time.dt.hour.values)
    ],
}
(OUTPUT_ROOT / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)
print(json.dumps(metadata, indent=2))
